In [ ]:
import nltk
import shutil
import os

# -- Fix broken/corrupt punkt installs --
nltk_data_path = os.path.expanduser('~/nltk_data/tokenizers/punkt')
if os.path.exists(nltk_data_path):
    shutil.rmtree(nltk_data_path, ignore_errors=True)
nltk.download('punkt')
nltk.download('punkt_tab') # Add this line to download the missing resource

from nltk.tokenize import sent_tokenize
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import warnings

warnings.filterwarnings('ignore')
print("✓ Libraries and NLTK data ready!")

✓ Libraries and NLTK data ready!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
class AdvancedTextSummarizer:
    def __init__(self, model_name='facebook/bart-large-cnn'):
        print(f"🔄 Loading model: {model_name}...")
        self.device = 0 if torch.cuda.is_available() else -1
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        if torch.cuda.is_available():
            self.model = self.model.cuda()
        self.summarizer = pipeline(
            "summarization",
            model=self.model,
            tokenizer=self.tokenizer,
            device=self.device
        )
        print(f"✓ Model loaded on {'GPU' if self.device==0 else 'CPU'}")

    def count_tokens(self, text):
        tokens = self.tokenizer.encode(text, truncation=False)
        return len(tokens)

    def split_into_chunks(self, text, max_tokens=1000):
        sentences = sent_tokenize(text)
        chunks, current_chunk, current_tokens = [], "", 0
        for sentence in sentences:
            sentence_tokens = self.count_tokens(sentence)
            if current_tokens + sentence_tokens <= max_tokens:
                current_chunk += " " + sentence
                current_tokens += sentence_tokens
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = sentence
                current_tokens = sentence_tokens
        if current_chunk:
            chunks.append(current_chunk.strip())
        return chunks

    def summarize_short_text(self, text, max_length=130, min_length=30):
        try:
            summary = self.summarizer(
                text, max_length=max_length, min_length=min_length, do_sample=False, truncation=True
            )
            return summary[0]['summary_text']
        except Exception as e:
            return f"Error in summarization: {str(e)}"

    def summarize_long_text(self, text, max_length=150, min_length=50):
        print("📄 Processing large document...")
        chunks = self.split_into_chunks(text, max_tokens=900)
        print(f"✓ Document split into {len(chunks)} chunks")
        chunk_summaries = []
        for i, chunk in enumerate(chunks):
            print(f"⚙️  Summarizing chunk {i+1}/{len(chunks)}...")
            summary = self.summarize_short_text(chunk, max_length, min_length)
            chunk_summaries.append(summary)
        combined_summary = " ".join(chunk_summaries)
        if self.count_tokens(combined_summary) > 900:
            print("🔄 Summarizing the combined summary...")
            return self.summarize_short_text(combined_summary, max_length*2, min_length)
        return combined_summary

    def summarize(self, text, max_length=150, min_length=50, auto_detect=True):
        token_count = self.count_tokens(text)
        print(f"📊 Token count: {token_count}")
        if auto_detect and token_count > 900:
            return self.summarize_long_text(text, max_length, min_length)
        else:
            return self.summarize_short_text(text, max_length, min_length)

print("✓ Summarizer class defined!")

✓ Summarizer class defined!


In [ ]:




def get_user_input():
    print("=" * 70)
    print("ADVANCED NLP TEXT SUMMARIZER")
    print("=" * 70)
    print("\n📝 Enter or paste your text below (press Enter twice when done):\n")
    lines = []
    empty_line_count = 0
    while True:
        try:
            line = input()
            if line.strip():
                lines.append(line)
                empty_line_count = 0
            else:
                empty_line_count += 1
                if empty_line_count >= 1:
                    break
        except EOFError:
            break
    text = " ".join(lines)
    return text

user_text = get_user_input()
if not user_text.strip():
    print("❌ No text provided! Exiting.")
    exit()

print("\n" + "=" * 70)
print("INPUT ANALYSIS")
print("=" * 70)
print(f"Characters: {len(user_text)}")
print(f"Words: {len(user_text.split())}")
print(f"Sentences: {len(sent_tokenize(user_text))}")
print("=" * 70)

summarizer = AdvancedTextSummarizer(model_name='sshleifer/distilbart-cnn-12-6')
print("\n🔄 Generating summary...\n")
summary = summarizer.summarize(user_text, max_length=100, min_length=35)
print("\n" + "=" * 70)
print("📋 SUMMARY")
print("=" * 70)
print(summary)
print("=" * 70)



ADVANCED NLP TEXT SUMMARIZER

📝 Enter or paste your text below (press Enter twice when done):

Climate change represents one of the most pressing challenges facing humanity in the twenty-first century, with far-reaching implications for ecosystems, economies, and societies worldwide. The scientific consensus is overwhelming that global temperatures are rising due to increased concentrations of greenhouse gases in the atmosphere, primarily carbon dioxide from burning fossil fuels. The effects of climate change are already visible across the planet. Polar ice caps and glaciers are melting at unprecedented rates, contributing to rising sea levels that threaten coastal communities and low-lying island nations. Extreme weather events, including hurricanes, droughts, heat waves, and floods, are becoming more frequent and intense. Ocean temperatures are increasing, leading to coral bleaching and disruption of marine ecosystems. Changes in precipitation patterns affect agriculture and water av

Device set to use cuda:0


✓ Model loaded on GPU

🔄 Generating summary...

📊 Token count: 440

📋 SUMMARY
 Climate change represents one of the most pressing challenges facing humanity in the twenty-first century . The scientific consensus is overwhelming that global temperatures are rising due to increased concentrations of greenhouse gases in the atmosphere . The effects of climate change are already visible across the planet .
